# Student T-Test

Statistical significance testing on model results.

In [9]:
import pandas as pd
import numpy as np
import os
import json
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

In [10]:
dataset = "Analgesics-induced_acute_liver_failure"

with open(os.path.join("/home/hsdslab/Documents/Csabi/Pharma_crossval/DeepCausalPV-master-main","dat",dataset,"proc", "all_results.json"), "r") as f:
    results = json.load(f)

In [11]:
metrics = {
    "auc": 0,
    "precision": 1,
    "recall": 2,
    "f1": 3,
    "accuracy": 4,
    "ece": 5
}

In [16]:
def calc_student(model1, model2, metric):
    model1_vals = []
    model2_vals = []
    
    for key in results.keys():
        model1_vals.append(results[key][model1][metrics[metric]])
        model2_vals.append(results[key][model2][metrics[metric]])
        
    model1_vals = np.array(model1_vals)
    model2_vals = np.array(model2_vals)
    
    if metric=="ece":
        model1_vals = model1_vals*-1
        model2_vals = model2_vals*-1
    
    # Calculate differences between paired observations
    differences = model1_vals - model2_vals
    
    # Calculate mean difference and standard deviation
    mean_diff = np.mean(differences)
    std_diff = np.std(differences, ddof=1)  # ddof=1 for sample std
    
    # Number of paired observations
    n = len(differences)
    
    # Calculate t-statistic
    t_stat = mean_diff / (std_diff / np.sqrt(n))
    
    # Degrees of freedom
    df = n - 1
    
    # Calculate one-sided p-value (testing if model1 > model2)
    p_value = 1 - stats.t.cdf(t_stat, df)
    
    return p_value

In [17]:
def get_pvaluematrix(metric):
    model_names = list(results[list(results.keys())[0]].keys())
    pvalue_matrix = np.zeros((len(model_names), len(model_names)))

    for i in range(len(model_names)):
        for j in range(len(model_names)):
            if i==j:
                pvalue_matrix[i, j] = 1
            else:
                pvalue_matrix[i, j] = calc_student(model_names[i], model_names[j], metric)
                
    return pvalue_matrix, model_names

In [26]:
def get_heatmap(metric):
    pvalue_matrix, model_names = get_pvaluematrix(metric)
    colors = ['red', 'green']
    cmap = ListedColormap(colors)

    # Create the heatmap
    plt.figure(figsize=(10, 10))
    # Use vmin and vmax to set the threshold at 0.05
    sns.heatmap(pvalue_matrix, 
            annot=True, 
            xticklabels=model_names, 
            yticklabels=model_names, 
            cmap=cmap, 
            cbar=False,
            vmin=0, 
            vmax=0.05,
            center=0.05,
            linewidth=0.05)  # Center the colormap at 0.05

    plt.title(f"Student's t-test p-values for {dataset}, {metric}")
    plt.tight_layout()
    plt.savefig(os.path.join("/home/hsdslab/Documents/Csabi/Pharma_crossval/DeepCausalPV-master-main/dat",dataset,"results/pvalue_matrices", f"{metric}_pvalues.png"))
    plt.close()

In [27]:
for metric in metrics.keys():
    get_heatmap(metric)